In [1]:
import os
os.environ["RELBENCH_CACHE_DIR"] = os.path.expanduser("~/scratch/relbench")
!echo $RELBENCH_CACHE_DIR

/lfs/ampere2/0/jonasds/scratch/relbench


In [24]:
import pandas as pd
import numpy as np
from relbench.tasks import get_task

task = get_task("rel-amazon", "user-item-purchase")
task.get_table("train").df.head()

,timestamp,customer_id,product_id
0,2014-04-03,393192,"[80225, 12439]"
1,2014-04-03,149856,"[38972, 29137, 422124, 61124, 80784]"
2,2014-04-03,384140,"[40744, 129519, 410405]"
3,2014-04-03,747159,"[26162, 22215, 26195]"
4,2014-04-03,318496,[12321]


In [23]:
# Get the task tables
train_table = task.get_table("train")
val_table = task.get_table("val")
test_table = task.get_table("test")

col = task.src_entity_col
print(col)

for customer_id in val_table.df[col].unique():
    n_occurences_in_train = (train_table.df[col] == customer_id).sum()
    n_occurences_in_val = (val_table.df[col] == customer_id).sum()
    n_occurences_in_test = (test_table.df[col] == customer_id).sum()
    print(customer_id, n_occurences_in_train, n_occurences_in_val, n_occurences_in_test)


driverId
39 0 1 0
29 14 1 0
34 9 1 0
10 3 1 0
18 1 1 0
41 2 1 0
3 3 1 0
38 0 1 0
16 3 1 0
26 0 1 0
21 12 1 0
37 0 1 0
14 8 1 0
30 4 1 0
17 5 1 0
32 0 1 0
31 1 1 0
22 8 1 0
12 2 1 0
20 9 1 0
36 4 1 0
23 0 1 0
40 4 1 0
24 4 1 0
13 11 1 0
7 4 1 0
1 5 1 0


In [18]:
pred_isin = np.array([
    [1,1,0,0,1,0,0,0,0,1],
    [0,0,0,0,0,0,0,0,0,0],
    [1,1,1,1,1,1,1,1,1,1],
    [0,0,0,0,0,0,1,0,0,0],
])

num_dst_nodes = pred_isin.shape[-1]
dst_count = pred_isin.sum(axis=-1)

cum = np.cumsum(pred_isin, axis=-1)

valid_mask = (dst_count > 0) & (dst_count < num_dst_nodes)


print((cum * (pred_isin == 0)).sum(axis=-1))

print((dst_count * (num_dst_nodes - dst_count)))

auc = (1 / (dst_count * (num_dst_nodes - dst_count))[valid_mask]) * (cum * (pred_isin == 0)).sum(axis=-1)[valid_mask]

print(valid_mask)
print(auc)

[16  0  0  3]
[24  0  0  9]
[ True False False  True]
[0.66666667 0.33333333]


In [ ]:
# Load the database
db = task.get_dataset().get_db()

print("="*80)
print("DATABASE TABLES")
print("="*80)
for table_name, table in db.table_dict.items():
    print(f"{table_name}: {len(table.df)} rows")
print()

# Examine the results table (the main source for this task)
results_df = db.table_dict["results"].df
print("Results table columns:", results_df.columns.tolist())
print("\nResults table sample:")
print(results_df.head())
print(f"\nResults table date range: {results_df['date'].min()} to {results_df['date'].max()}")


In [ ]:
# Get the task tables
train_table = task.get_table("train")
val_table = task.get_table("val")
test_table = task.get_table("test")

print("="*80)
print("TASK TABLES")
print("="*80)
print(f"Train: {len(train_table.df)} rows")
print(f"Val: {len(val_table.df)} rows")
print(f"Test: {len(test_table.df)} rows")
print("\nTrain table columns:", train_table.df.columns.tolist())
print("\nTrain table sample:")
print(train_table.df.head(10))


In [ ]:
# Check for data quality issues
print("="*80)
print("DATA QUALITY CHECKS")
print("="*80)

train_df = train_table.df

# Check for null values
print("Null values:")
print(train_df.isnull().sum())
print()

# Check race lists
print("Race list statistics:")
print(f"Total rows: {len(train_df)}")
print(f"Rows with empty race lists: {train_df['raceId'].apply(lambda x: len(x) == 0).sum()}")
print(f"Rows with non-empty race lists: {train_df['raceId'].apply(lambda x: len(x) > 0).sum()}")

# Distribution of list lengths
list_lengths = train_df['raceId'].apply(len)
print(f"\nRace list length statistics:")
print(f"  Mean: {list_lengths.mean():.2f}")
print(f"  Median: {list_lengths.median():.2f}")
print(f"  Max: {list_lengths.max()}")
print(f"  Min: {list_lengths.min()}")
print(f"\nDistribution of list lengths:")
print(list_lengths.value_counts().sort_index().head(20))


In [ ]:
# CRITICAL: Manual verification of temporal logic
print("="*80)
print("TEMPORAL LOGIC VERIFICATION")
print("="*80)

# Pick a specific timestamp and driver to manually verify
sample_row = train_df[train_df['raceId'].apply(len) > 0].iloc[5]
timestamp = sample_row['date']
driver_id = sample_row['driverId']
predicted_races = sample_row['raceId']

print(f"\nSample verification:")
print(f"  Timestamp: {timestamp}")
print(f"  Driver ID: {driver_id}")
print(f"  Predicted race IDs: {predicted_races}")
print(f"  Number of races: {len(predicted_races)}")

# Manually query the results table to verify
window_start = timestamp
window_end = timestamp + task.timedelta

actual_races = results_df[
    (results_df['driverId'] == driver_id) &
    (results_df['date'] > window_start) &
    (results_df['date'] <= window_end)
]['raceId'].unique().tolist()

print(f"\nManual verification:")
print(f"  Window: {window_start} < date <= {window_end}")
print(f"  Actual race IDs from results table: {sorted(actual_races)}")
print(f"  Task predicted race IDs: {sorted(predicted_races)}")
print(f"  Match: {sorted(actual_races) == sorted(predicted_races)}")

# Check for temporal leakage (no races at or before timestamp)
if len(predicted_races) > 0:
    races_at_or_before = results_df[
        (results_df['driverId'] == driver_id) &
        (results_df['date'] <= window_start) &
        (results_df['raceId'].isin(predicted_races))
    ]
    print(f"\n  DATA LEAKAGE CHECK:")
    print(f"  Races at or before timestamp: {len(races_at_or_before)}")
    if len(races_at_or_before) > 0:
        print("  ⚠️  WARNING: TEMPORAL LEAKAGE DETECTED!")
        print(races_at_or_before[['date', 'raceId', 'driverId']])
    else:
        print("  ✓ No temporal leakage detected")


In [ ]:
# Comprehensive temporal verification on multiple samples
print("="*80)
print("COMPREHENSIVE TEMPORAL VERIFICATION (10 SAMPLES)")
print("="*80)

num_mismatches = 0
num_leakages = 0
samples_checked = 0

# Test on 10 random samples with non-empty race lists
sample_indices = train_df[train_df['raceId'].apply(len) > 0].sample(min(10, len(train_df))).index

for idx in sample_indices:
    row = train_df.loc[idx]
    timestamp = row['date']
    driver_id = row['driverId']
    predicted_races = row['raceId']
    
    window_start = timestamp
    window_end = timestamp + task.timedelta
    
    # Get actual races from database
    actual_races = results_df[
        (results_df['driverId'] == driver_id) &
        (results_df['date'] > window_start) &
        (results_df['date'] <= window_end)
    ]['raceId'].unique().tolist()
    
    match = sorted(actual_races) == sorted(predicted_races)
    samples_checked += 1
    
    if not match:
        num_mismatches += 1
        print(f"\n❌ MISMATCH at index {idx}:")
        print(f"   Timestamp: {timestamp}, Driver: {driver_id}")
        print(f"   Expected: {sorted(actual_races)}")
        print(f"   Got: {sorted(predicted_races)}")
    
    # Check for temporal leakage
    if len(predicted_races) > 0:
        races_at_or_before = results_df[
            (results_df['driverId'] == driver_id) &
            (results_df['date'] <= window_start) &
            (results_df['raceId'].isin(predicted_races))
        ]
        if len(races_at_or_before) > 0:
            num_leakages += 1
            print(f"\n⚠️  TEMPORAL LEAKAGE at index {idx}:")
            print(f"   Timestamp: {timestamp}, Driver: {driver_id}")
            print(f"   Leaked races: {races_at_or_before['raceId'].tolist()}")

print(f"\n{'='*80}")
print("VERIFICATION SUMMARY")
print("="*80)
print(f"Samples checked: {samples_checked}")
print(f"Mismatches: {num_mismatches}")
print(f"Temporal leakages: {num_leakages}")

if num_mismatches == 0 and num_leakages == 0:
    print("\n✅ ALL CHECKS PASSED - Task appears to be correctly implemented!")
else:
    print("\n❌ ISSUES FOUND - Task may have errors!")


In [ ]:
# Additional check: Verify race entities exist in races table
print("="*80)
print("ENTITY CONSISTENCY CHECK")
print("="*80)

races_table = db.table_dict["races"].df
all_valid_race_ids = set(races_table['raceId'].unique())

# Collect all race IDs from predictions
all_predicted_race_ids = set()
for race_list in train_df['raceId']:
    all_predicted_race_ids.update(race_list)

print(f"Valid race IDs in races table: {len(all_valid_race_ids)}")
print(f"Unique race IDs in predictions: {len(all_predicted_race_ids)}")

# Check if all predicted races exist
invalid_races = all_predicted_race_ids - all_valid_race_ids
if len(invalid_races) > 0:
    print(f"\n❌ Found {len(invalid_races)} invalid race IDs in predictions:")
    print(f"   {list(invalid_races)[:10]}...")
else:
    print("\n✅ All predicted race IDs exist in the races table")

# Check drivers
drivers_table = db.table_dict["drivers"].df
all_valid_driver_ids = set(drivers_table['driverId'].unique())
all_predicted_driver_ids = set(train_df['driverId'].unique())

print(f"\nValid driver IDs in drivers table: {len(all_valid_driver_ids)}")
print(f"Unique driver IDs in predictions: {len(all_predicted_driver_ids)}")

invalid_drivers = all_predicted_driver_ids - all_valid_driver_ids
if len(invalid_drivers) > 0:
    print(f"\n❌ Found {len(invalid_drivers)} invalid driver IDs in predictions:")
    print(f"   {list(invalid_drivers)[:10]}...")
else:
    print("\n✅ All predicted driver IDs exist in the drivers table")
